# Stage 6.5 GSA: variance-based sensitivity of the Phase 1 engine

Thin driver (spec section 9: no physics in notebooks). The analysis itself runs via
```
python scripts/gsa_study.py
```
which writes the JSON records to `results/gsa/` (tracked copies under
`docs/decisions/adr0033-gsa-study-*.json`) and the figures to `docs/figures/`.
This notebook only reloads those records, regenerates the figures, and tabulates
the design-level indices. Method and every methodological decision: **ADR-0033**;
full write-up: `docs/decisions/adr0033-gsa-study.md`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

payloads = {
    slug: json.loads((REPO_ROOT / "results" / "gsa" / f"{slug}_gsa.json").read_text())
    for slug in ("kp58_8_matrix", "kp60_0_matrix")
}
companions = json.loads(
    (REPO_ROOT / "results" / "gsa" / "kp58_8_matrix_companions_gsa.json").read_text()
)
for slug, p in payloads.items():
    print(slug, "| levels:", [lv["level_m"] for lv in p["levels"]],
          "| scenario invariance:",
          p["scenario_invariance_check"]["loading_bit_identical_across_scenarios"])

In [ ]:
# Regenerate every figure from the JSON records (same code path as the script).
import gsa_study as gsa

for slug, payload in payloads.items():
    gsa._plot_section(payload, slug)
gsa._fig_companions(companions, payloads["kp58_8_matrix"])

In [ ]:
from IPython.display import Image, display

for slug in payloads:
    for kind in ("indices", "levels", "interaction", "convergence"):
        display(Image(str(REPO_ROOT / "docs" / "figures" / f"gsa_{kind}_{slug}.png")))
display(Image(str(REPO_ROOT / "docs" / "figures" / "gsa_companions.png")))

In [ ]:
# Design-level index table (final rung, replicate means with 95% t-CIs).
import pandas as pd

rows = []
for slug, p in payloads.items():
    design = p["levels"][1]
    for qoi_key, qoi in design["qois"].items():
        final = qoi["rungs"][-1]
        for j, name in enumerate(p["input_names"]):
            rows.append({
                "section": p["cross_section_id"], "qoi": qoi_key, "input": name,
                "S": final["S_mean"][j],
                "S_lo": final["S_lo"][j], "S_hi": final["S_hi"][j],
                "ST": final["ST_mean"][j],
                "ST_lo": final["ST_lo"][j], "ST_hi": final["ST_hi"][j],
            })
df = pd.DataFrame(rows)
df[df.qoi == "trans_indicator"].sort_values(["section", "ST"], ascending=[True, False])